In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

In [ ]:
from tensorflow.python.keras.utils.np_utils import to_categorical

data = keras.datasets.fashion_mnist
(x_train, y_train), (x_test, y_test) = data.load_data()

x_train = x_train.reshape(-1, 28, 28, 1).astype("float32") / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype("float32") / 255.0
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

In [ ]:
def build_lenet5():
    model = keras.Sequential([
        layers.Conv2D(6, kernel_size=5, activation='relu', input_shape=(28, 28, 1)),
        layers.AvgPool2D(pool_size=2),
        layers.Conv2D(16, kernel_size=5, activation='relu'),
        layers.AvgPool2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(120, activation='relu', name="feature_layer"),
        layers.Dense(84, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

lenet5 = build_lenet5()
lenet5.fit(x_train, y_train, epochs=10, validation_data=(x_test, y_test))

In [ ]:
for i, layer in enumerate(lenet5.layers):
    print(f"Шар {i}: {layer.name} - Тип: {type(layer).__name__} - Вихідна форма: {layer.output.shape}")

In [ ]:
def build_siamese_network():
    input_shape = (28, 28, 1)
    base_network = keras.Sequential([
        layers.Conv2D(64, kernel_size=3, activation='relu', input_shape=input_shape),
        layers.MaxPooling2D(pool_size=2),
        layers.Conv2D(128, kernel_size=3, activation='relu'),
        layers.MaxPooling2D(pool_size=2),
        layers.Flatten(),
        layers.Dense(128, activation='relu')
    ])

    input_a = keras.Input(shape=input_shape)
    input_b = keras.Input(shape=input_shape)
    encoded_a = base_network(input_a)
    encoded_b = base_network(input_b)

    distance = layers.Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([encoded_a, encoded_b])
    output = layers.Dense(1, activation='sigmoid')(distance)

    model = keras.Model(inputs=[input_a, input_b], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

siamese_network = build_siamese_network()

In [ ]:
# Виконати перед отриманням ознак
lenet5.predict(np.expand_dims(x_test[0], axis=0))

In [ ]:
feature_extractor = keras.Model(inputs=lenet5.input, outputs=lenet5.layers[-2].output)
features = feature_extractor.predict(x_test)

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
reduced_features = tsne.fit_transform(features)

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(reduced_features[:, 0], reduced_features[:, 1], c=y_test, cmap='tab10', alpha=0.7)
plt.colorbar()
plt.title("t-SNE visualization of Fashion-MNIST features")
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# Загрузка и предобработка данных Fashion-MNIST
(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

# Определение модели LeNet-5
model = models.Sequential([
    layers.Conv2D(6, (5, 5), activation='tanh', padding='same', input_shape=(28, 28, 1)),
    layers.AvgPool2D((2, 2), strides=2),
    layers.Conv2D(16, (5, 5), activation='tanh', padding='valid'),
    layers.AvgPool2D((2, 2), strides=2),
    layers.Flatten(),
    layers.Dense(120, activation='tanh'),
    layers.Dense(84, activation='tanh'),
    layers.Dense(10, activation='softmax')
])

# Компиляция модели
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(x_train, y_train, epochs=10, batch_size=64, validation_data=(x_test, y_test))

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)
print(f'Точность на тестовых данных: {test_acc}')

model.build(input_shape=(None, 28, 28, 1))
_ = model.predict(np.expand_dims(x_test[0], axis=0))

In [ ]:
sample_size = 5000
indices = np.random.choice(x_test.shape[0], sample_size, replace=False)
features_sample = features[indices]
labels_sample = np.argmax(y_test, axis=1)[indices]

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
features_2d = tsne.fit_transform(features_sample)



In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=labels_sample, cmap='jet', alpha=0.6)
plt.colorbar(scatter, label="Классы")
plt.title("Визуализация признаков Fashion-MNIST с помощью t-SNE")
plt.xlabel("Компонент 1")
plt.ylabel("Компонент 2")
plt.show()